# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.date_published if hasattr(metadata, 'date_published') else 'N/A'}\n")
print(f"License: {metadata.license if hasattr(metadata, 'license') else 'N/A'}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each record set has a unique `@id`. We enumerate these below using the dataset metadata. For each record set, we also list the available fields and their associated `@id`.

In [ ]:
# List all available record sets with their @id and fields
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List the fields associated with this record set
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (id: {f.id}, dataType: {getattr(f, 'data_type', 'N/A')})")
    print()
if not record_set_ids:
    print("No record sets found. Please verify the dataset schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> Note: For this dataset, we use the first available record set as an example. Replace the record set ID and field IDs as needed for your analysis.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}. Error: {e}")

# Example: show columns for the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns in first record set ({example_record_set_id}):")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> Below, we pick a numeric field and a categorical/group field from the first record set. You may change the field IDs to match those relevant in your analysis.

In [ ]:
# Example EDA with the loaded DataFrame
import numpy as np

# Set these after inspecting your record set fields
df = dataframes[example_record_set_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in this table: {numeric_fields}")
if numeric_fields:
    numeric_field = numeric_fields[0]  # Example: first numeric field
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0

    # Simple filter: greater than mean
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
    display(filtered_df[[numeric_field]].head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' (first 5 records):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by first non-numeric field, if available
    group_fields = [col for col in df.columns if col not in numeric_fields]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Mean statistics grouped by '{group_field}':")
        display(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Example: Plot histogram and boxplot for a chosen numeric field and a bar plot grouping by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_fields:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

    # If grouped/categorical field available
    if group_fields:
        counts = df[group_field].value_counts().nlargest(10)
        plt.figure(figsize=(8,4))
        sns.barplot(x=counts.values, y=counts.index, orient='h')
        plt.title(f'Top 10 {group_field} counts')
        plt.xlabel('Count')
        plt.ylabel(group_field)
        plt.show()
else:
    print("No suitable fields found for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and inspected the Croissant schema, viewing the dataset record sets, fields and sample data.
- Basic filtering, normalization, and grouping EDA steps were applied to numeric data if available.
- Example visualizations illustrated data distributions and category counts.

Continue your analysis by exploring additional record sets, fields, and custom processing relevant to your research objectives.